# Script 1 — Data Cleaning
**Project:** Do bad years on farmland push clearing into the forest? San Martín, Peru (2001–2024)

**The rule:** read from `raw_data/`, clean, write to `clean_data/`. Never modify raw data.

**Input** (made by `0_gee_extract.js` in Earth Engine):
- `raw_data/SM_loss_by_unit_year.csv` — forest loss (ha) per province per year
- `raw_data/SM_condition_by_unit_year.csv` — farmland greenness, rain, drought, farmland share per province per year

**Output:**
- `clean_data/sm_province_year_clean.csv` — one table, one row per province per year

**My steps in plain words:**
0. Connect Colab to my Google Drive
1. Go to my project folder
2. Read the two raw tables and look at them
3. Fix names and types: rename the province column, make `year` a whole number
4. Check that every province has all 24 years, with no duplicates
5. Check that the values make sense (no negative hectares, greenness between 0 and 1, etc.)
6. Flag the rows with no greenness value, but keep them
7. Join the two tables by province and year
8. Save the clean table to `clean_data/`

## Step 0 — Connect Google Drive (Python only)
Colab cannot see my Drive until I connect it. This one cell runs in **Python**; everything else runs in **R**.
1. **Runtime → Change runtime type → Python 3 → Save**
2. Run the cell below. Click **Allow** when Google asks.
3. **Runtime → Change runtime type → R → Save**
4. Go on with Step 1. Do not run this cell again in R.

In [1]:
# Run this cell with a Python runtime only.
# Switch back to R before running any cell below.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 1 — Go to my project folder
This cell moves into my project folder in Drive and checks that the two raw files are there.

In [1]:
# --- Step 1: go to my project folder -----------------------------------------
project_name <- "san_martin_cocoa_clearing"
setwd(file.path("/content/drive/MyDrive", project_name))
getwd()                      # should end with san_martin_cocoa_clearing
list.files("raw_data/")      # should show the two SM_... files

[1] "/content/drive/MyDrive/san_martin_cocoa_clearing"

[1] "SM_condition_by_unit_year.csv" "SM_loss_by_unit_year.csv"

## Step 2 — Read the two raw tables
`read_csv()` reads each file into a data frame. `head()` shows the first rows. `nrow()` counts rows.
I expect 240 rows in each table: 10 provinces × 24 years.

In [2]:
# --- Step 2: read the raw data -------------------------------------------------
library(readr)
library(dplyr)

loss_raw <- read_csv("raw_data/SM_loss_by_unit_year.csv")
cond_raw <- read_csv("raw_data/SM_condition_by_unit_year.csv")

head(loss_raw)
head(cond_raw)

cat("Rows in loss table:     ", nrow(loss_raw), "\n")
cat("Rows in condition table:", nrow(cond_raw), "\n")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Rows: 240 Columns: 3
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): ADM2_NAME
dbl (2): year, loss_ha

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 240 Columns: 8
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): ADM2_NAME
dbl (7): year, evi_dry, evi_dry_n, evi_year, rain_mm, pdsi, farm_share

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


ADM2_NAME,year,loss_ha
<chr>,<dbl>,<dbl>
Bellavista,2001,2304.3142
Lamas,2001,1699.4542
Moyobamba,2001,578.1338
Picota,2001,1460.0292
Rioja,2001,334.4797
San Martín,2001,1480.6977


ADM2_NAME,year,evi_dry,evi_dry_n,evi_year,rain_mm,pdsi,farm_share
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Bellavista,2001,0.4098429,6.784992,0.4422930,1476.542,-0.63705293,0.020240834
Lamas,2001,0.4827969,7.610617,0.5016678,1549.522,0.74478268,0.005956626
Moyobamba,2001,0.3525979,6.210932,0.4227123,1382.019,1.97714659,0.003351637
Picota,2001,0.3994040,6.853595,0.4268787,1096.326,0.05338025,0.024675799
Rioja,2001,0.4055991,4.858707,0.4498465,1254.988,2.37038020,0.035894886
San Martín,2001,0.4447118,7.809990,0.4625420,1621.348,-0.03379852,0.010480332


Rows in loss table:      240 
Rows in condition table: 240 


## Step 3 — Fix names and types
Two small problems in the raw tables:
- The province column is called `ADM2_NAME`. That is the code name from the boundary map. I rename it to `province`, so the table is easy to read.
- `year` comes as a decimal number (2001.0). A year is a whole number, so I change it with `as.integer()`. This also makes the join in Step 7 safe: both tables must have the same kind of `year`.

In [3]:
# --- Step 3: rename the province column and make year a whole number ----------
loss <- loss_raw %>%
  rename(province = ADM2_NAME) %>%
  mutate(year = as.integer(year))

cond <- cond_raw %>%
  rename(province = ADM2_NAME) %>%
  mutate(year = as.integer(year))

head(loss)
head(cond)

province,year,loss_ha
<chr>,<int>,<dbl>
Bellavista,2001,2304.3142
Lamas,2001,1699.4542
Moyobamba,2001,578.1338
Picota,2001,1460.0292
Rioja,2001,334.4797
San Martín,2001,1480.6977


province,year,evi_dry,evi_dry_n,evi_year,rain_mm,pdsi,farm_share
<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Bellavista,2001,0.4098429,6.784992,0.4422930,1476.542,-0.63705293,0.020240834
Lamas,2001,0.4827969,7.610617,0.5016678,1549.522,0.74478268,0.005956626
Moyobamba,2001,0.3525979,6.210932,0.4227123,1382.019,1.97714659,0.003351637
Picota,2001,0.3994040,6.853595,0.4268787,1096.326,0.05338025,0.024675799
Rioja,2001,0.4055991,4.858707,0.4498465,1254.988,2.37038020,0.035894886
San Martín,2001,0.4447118,7.809990,0.4625420,1621.348,-0.03379852,0.010480332


## Step 4 — Check that every province has all 24 years
`count()` counts rows per province. Each province should have 24 (2001 to 2024).
Then I check there are no duplicate rows (the same province and year twice).

In [4]:
# --- Step 4: completeness and duplicates --------------------------------------
# Rows per province: every province should show n = 24
loss %>% count(province)
cond %>% count(province)

# Duplicates: the same province + year more than once. Should print 0 for both.
dup_loss <- loss %>% count(province, year) %>% filter(n > 1)
dup_cond <- cond %>% count(province, year) %>% filter(n > 1)
cat("Duplicate rows in loss table:     ", nrow(dup_loss), "\n")
cat("Duplicate rows in condition table:", nrow(dup_cond), "\n")

province,n
<chr>,<int>
Bellavista,24
El Dorado,24
Huallaga,24
Lamas,24
Mariscal Cáceres,24
Moyobamba,24
Picota,24
Rioja,24
San Martín,24


province,n
<chr>,<int>
Bellavista,24
El Dorado,24
Huallaga,24
Lamas,24
Mariscal Cáceres,24
Moyobamba,24
Picota,24
Rioja,24
San Martín,24


Duplicate rows in loss table:      0 
Duplicate rows in condition table: 0 


## Step 5 — Check that the values make sense
`summary()` shows the minimum, maximum and average of each column, and how many values are missing (`NA's`).
What I expect:
- `loss_ha`: 0 or more (you cannot lose negative forest)
- `evi_dry`, `evi_year`: between 0 and 1 (greenness index)
- `evi_dry_n`: between 0 and 8 (clear images in June–September)
- `rain_mm`: roughly 800 to 3,000 mm per year in San Martín
- `pdsi`: roughly −10 to +10 (below 0 = drier than normal)
- `farm_share`: between 0 and 1

In [5]:
# --- Step 5: value ranges ------------------------------------------------------
summary(loss)
summary(cond)

# Count values outside the expected range. Each line should print 0.
cat("Negative loss values:          ", nrow(filter(loss, loss_ha < 0)), "\n")
cat("Greenness outside 0-1:         ", nrow(filter(cond, evi_dry < 0 | evi_dry > 1)), "\n")
cat("Farm share outside 0-1:        ", nrow(filter(cond, farm_share < 0 | farm_share > 1)), "\n")
cat("Rain below 0:                  ", nrow(filter(cond, rain_mm < 0)), "\n")

      province        year         loss_ha       
 Length   :240   Min.   :2001   Min.   :  334.5  
 N.unique : 10   1st Qu.:2007   1st Qu.: 1748.7  
 N.blank  :  0   Median :2012   Median : 2913.4  
 Min.nchar:  5   Mean   :2012   Mean   : 3192.3  
 Max.nchar: 16   3rd Qu.:2018   3rd Qu.: 4233.4  
                 Max.   :2024   Max.   :11837.6  

      province        year         evi_dry         evi_dry_n    
 Length   :240   Min.   :2001   Min.   :0.3454   Min.   :2.566  
 N.unique : 10   1st Qu.:2007   1st Qu.:0.3895   1st Qu.:5.492  
 N.blank  :  0   Median :2012   Median :0.4115   Median :6.318  
 Min.nchar:  5   Mean   :2012   Mean   :0.4103   Mean   :6.166  
 Max.nchar: 16   3rd Qu.:2018   3rd Qu.:0.4295   3rd Qu.:6.964  
                 Max.   :2024   Max.   :0.4844   Max.   :7.987  
                                NAs    :48       NAs    :48     
    evi_year         rain_mm          pdsi           farm_share       
 Min.   :0.3798   Min.   : 920   Min.   :-6.0887   Min.   :0.0000000  
 1st Qu.:0.4110   1st Qu.:1333   1st Qu.:-0.6422   1st Qu.:0.0009467  
 Median :0.4320   Median :1565   Median : 0.8142   Median :0.0046541  
 Mean   :0.4300   Mean   :1595   Mean   : 0.8960   Mean   :0.0103585  
 3rd Qu.:0.4466   3rd Qu.:1814   3rd Qu.: 2.2678   3rd Qu.:0.0202408  
 Max.   :0.5017   Max.   :2771   Max.   : 8.5371   Max

Negative loss values:           0 
Greenness outside 0-1:          0 
Farm share outside 0-1:         0 
Rain below 0:                   0 


## Step 6 — Flag the rows with no greenness, but keep them
Two provinces, **El Dorado** and **Mariscal Cáceres**, have no greenness values (`NA`).
The reason: the MODIS land cover map (500 m) found no stable farmland there (`farm_share` = 0). No farmland means no greenness to measure.

In the practice set I dropped rows with `NA`. Here I do **not** drop them, because their forest loss, rain and drought values are still good and useful.
Instead I add a column `evi_missing` (TRUE / FALSE), so later steps can choose to leave them out of the greenness analysis.

In [6]:
# --- Step 6: flag missing greenness -------------------------------------------
cond <- cond %>%
  mutate(evi_missing = is.na(evi_dry))

# Which provinces have missing greenness, and in how many years?
cond %>%
  filter(evi_missing) %>%
  count(province)

province,n
<chr>,<int>
El Dorado,24
Mariscal Cáceres,24


## Step 7 — Join the two tables by province and year
`left_join()` puts two tables side by side, matching rows that share the same `province` and `year`.
Example: the row "Tocache, 2012" in the loss table gets the greenness, rain and drought of "Tocache, 2012" from the condition table.
The result should still have 240 rows. If it has more or fewer, the match went wrong.

In [7]:
# --- Step 7: join loss and condition into one table ---------------------------
sm_clean <- loss %>%
  left_join(cond, by = c("province", "year"))

head(sm_clean)
cat("Rows after join:", nrow(sm_clean), "(expect 240)\n")
cat("Columns:", ncol(sm_clean), "\n")

province,year,loss_ha,evi_dry,evi_dry_n,evi_year,rain_mm,pdsi,farm_share,evi_missing
<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>
Bellavista,2001,2304.3142,0.4098429,6.784992,0.4422930,1476.542,-0.63705293,0.020240834,FALSE
Lamas,2001,1699.4542,0.4827969,7.610617,0.5016678,1549.522,0.74478268,0.005956626,FALSE
Moyobamba,2001,578.1338,0.3525979,6.210932,0.4227123,1382.019,1.97714659,0.003351637,FALSE
Picota,2001,1460.0292,0.3994040,6.853595,0.4268787,1096.326,0.05338025,0.024675799,FALSE
Rioja,2001,334.4797,0.4055991,4.858707,0.4498465,1254.988,2.37038020,0.035894886,FALSE
San Martín,2001,1480.6977,0.4447118,7.809990,0.4625420,1621.348,-0.03379852,0.010480332,FALSE


Rows after join: 240 (expect 240)
Columns: 10 


## Step 8 — Save the clean table to `clean_data/`
`write_csv()` saves the table. `list.files()` checks the file is there.

In [8]:
# --- Step 8: write the clean data ---------------------------------------------
write_csv(sm_clean, "clean_data/sm_province_year_clean.csv")
list.files("clean_data/")

[1] "sm_province_year_clean.csv"